# 🚗 Cars Production Agent — Capstone Project

**Agentic AI Hands-On Course** | Dr. Kanthi Kiran Sirra | Sr. AI Engineer

**Domain:** Automotive — Cars Specs, Pricing & Buying Guide

This agent answers questions about car specifications, pricing, features, performance, fuel efficiency, safety ratings, and buying advice across popular car brands and models.

---
### All 6 Mandatory Capabilities:
1. ✅ LangGraph StateGraph (memory → router → retrieve/skip/tool → answer → eval → save)
2. ✅ ChromaDB RAG (12 domain documents covering specs, price, safety, etc.)
3. ✅ Conversation memory (MemorySaver + thread_id)
4. ✅ Self-reflection (eval node with faithfulness scoring + retry loop)
5. ✅ Tool use (DuckDuckGo web search for latest car prices & news)
6. ✅ Deployment (Streamlit UI)
---

## My Capstone Plan

**Domain:** Automotive — Car Specifications, Pricing, Features & Buying Guide

**User:** Car buyers, enthusiasts, and researchers who want quick, accurate info on cars — specs, price, fuel economy, safety, and comparisons between models.

**Success looks like:** The agent answers at least 90% of car-related questions accurately using the knowledge base, routes live price/news queries to web search, and gracefully admits when info is outside its knowledge base.

**Tool I will add:** DuckDuckGo web search (`ddgs`) — useful for real-time car prices, newly launched models, and dealer offers that change frequently.

**Deployment choice:** Streamlit UI (chat interface)

---
## 0. Setup

In [ ]:
# ============================================================
# COLAB USERS ONLY — Uncomment if using Google Colab
# ============================================================
# !pip install langgraph langchain-groq langchain-core chromadb \
#              sentence-transformers ragas ddgs python-dotenv -q

# from google.colab import userdata
# import os
# os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

In [ ]:
# ── Section 0: Setup (Dynamic LLM) ─────────────────────────

import os
from dotenv import load_dotenv
load_dotenv()

# 🔹 Core imports (used in whole project)
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver
from typing import TypedDict, List
import chromadb
from sentence_transformers import SentenceTransformer
from importlib.metadata import version

# 🔹 Choose provider from .env
provider = os.getenv("LLM_PROVIDER", "openai").lower()

# 🔐 Check keys
openai_key = os.getenv("OPENAI_API_KEY", "")
groq_key   = os.getenv("GROQ_API_KEY", "")

print(f"LLM Provider: {provider}")
print(f"OpenAI Key:   {'✅ Loaded' if len(openai_key) > 10 else '❌ Missing'}")
print(f"Groq Key:     {'✅ Loaded' if len(groq_key) > 10 else '❌ Missing'}")
print(f"LangGraph:    {version('langgraph')}")

# 🤖 Initialize LLM
if provider == "openai":
    from langchain_openai import ChatOpenAI

    llm = ChatOpenAI(
        model="gpt-4o-mini",   # fast + cheap
        temperature=0
    )

    print("✅ Using OpenAI")

elif provider == "groq":
    from langchain_groq import ChatGroq

    llm = ChatGroq(
        model="llama-3.3-70b-versatile",
        temperature=0
    )

    print("✅ Using Groq")

else:
    raise ValueError("❌ Invalid LLM_PROVIDER. Use 'openai' or 'groq'")

# 🧪 Test LLM
try:
    r = llm.invoke("Say ready in one word.")
    print(f"LLM Test:     ✅ {r.content}")
except Exception as e:
    print(f"LLM Test:     ❌ Error → {e}")

---
## Part 1 — Domain Setup: Cars Knowledge Base

12 documents covering: sedans, SUVs, EVs, sports cars, fuel economy, safety ratings, pricing tiers, maintenance costs, towing capacity, and buying tips.

In [ ]:
DOCUMENTS = [
    {
        "id": "doc_001",
        "topic": "Toyota Camry — Specs & Pricing",
        "text": """The Toyota Camry is one of the best-selling mid-size sedans in the world.
Available trims: LE, SE, XSE, XLE, TRD, and Hybrid. The base LE starts at approximately $27,000 USD.
Engine options: 2.5L 4-cylinder (203 hp) or 3.5L V6 (301 hp). The Hybrid variant uses a 2.5L engine
paired with an electric motor for a combined 208 hp and achieves up to 51 MPG city / 53 MPG highway.
Fuel economy for the standard 4-cylinder: 28 MPG city / 39 MPG highway.
The Camry seats 5 passengers, has a 15.1 cubic foot trunk, and comes standard with Toyota Safety Sense 2.5+
(pre-collision system, adaptive cruise control, lane departure alert, automatic high beams).
NHTSA overall safety rating: 5 stars. IIHS Top Safety Pick+. Infotainment: 8-inch or 12.3-inch touchscreen,
Apple CarPlay, Android Auto. Warranty: 3 years/36,000 miles basic; 5 years/60,000 miles powertrain.
The Camry is known for exceptional reliability and low cost of ownership."""
    },
    {
        "id": "doc_002",
        "topic": "Tesla Model 3 — Electric Sedan Specs",
        "text": """The Tesla Model 3 is a premium all-electric sedan with three variants:
Standard Range RWD (~$40,240), Long Range AWD (~$47,240), and Performance AWD (~$53,240) USD prices before incentives.
Range: Standard — 272 miles; Long Range — 358 miles; Performance — 315 miles (EPA estimated).
0-60 mph: Standard — 5.8 sec; Long Range — 4.2 sec; Performance — 3.1 sec.
Top speed: up to 162 mph (Performance). Charging: supports Tesla Supercharger V3 (up to 250 kW),
adding ~170 miles in 15 minutes. Home charging on a 240V outlet adds ~37 miles per hour.
Interior: minimalist 15.4-inch touchscreen controls almost everything. Seats 5. Trunk: 15 cu ft + 2.7 cu ft frunk.
Autopilot standard (Traffic-Aware Cruise Control + Autosteer). Full Self-Driving available as subscription.
NHTSA: 5-star overall. IIHS Top Safety Pick+. Over-the-air software updates keep the car improving.
Federal tax credit: up to $7,500 (eligibility depends on income and trim). Recommended for tech-forward buyers
who drive long distances and have home charging."""
    },
    {
        "id": "doc_003",
        "topic": "Honda CR-V — Compact SUV Specs",
        "text": """The Honda CR-V is a top-selling compact SUV praised for practicality and reliability.
Trims: LX, EX, EX-L, Sport, Sport-L, Sport Touring. Starting price: approximately $31,895 USD (LX FWD).
Engine: 1.5L turbocharged 4-cylinder (190 hp, 179 lb-ft torque) mated to a CVT transmission.
Hybrid option: 2.0L Atkinson-cycle + electric motor (204 hp combined); up to 40 MPG city / 34 MPG highway.
Non-hybrid fuel economy: 28 MPG city / 34 MPG highway (FWD). AWD available on all trims.
Cargo space: 39.3 cu ft behind rear seats; 76.5 cu ft with rear seats folded — one of the largest in its class.
Towing capacity: up to 1,500 lbs (non-hybrid). Seats 5 adults comfortably.
Honda Sensing safety suite standard on all trims: collision mitigation, lane keeping assist, adaptive cruise,
road departure mitigation. NHTSA: 5-star overall. IIHS Top Safety Pick+.
The CR-V is ideal for families needing cargo space, good fuel economy, and proven reliability."""
    },
    {
        "id": "doc_004",
        "topic": "BMW 3 Series — Luxury Sports Sedan",
        "text": """The BMW 3 Series is the benchmark luxury compact sports sedan. Starting at around $45,000 USD (330i).
Engine options: 2.0L turbocharged inline-4 (255 hp) in 330i; 3.0L inline-6 (382 hp) in M340i;
plug-in hybrid (288 hp combined) in 330e. All-wheel drive (xDrive) available on all variants.
Performance: 330i — 0-60 in 5.6 sec; M340i — 0-60 in 4.2 sec.
Fuel economy: 330i — 26 MPG city / 36 MPG highway. M340i — 25 MPG city / 33 MPG highway.
Driving dynamics are class-leading — rear-wheel drive bias, perfect 50:50 weight distribution, adaptive M suspension.
Infotainment: BMW iDrive 8 with 14.9-inch curved touchscreen, wireless Apple CarPlay/Android Auto.
Seats 5 (tight in rear). Trunk: 17 cubic feet. Warranty: 4 years/50,000 miles (bumper-to-bumper).
Safety: BMW's Driving Assistance Professional package includes lane change assist, evasion aid, cross-traffic alert.
Best suited for driving enthusiasts who want a premium feel and sporty dynamics."""
    },
    {
        "id": "doc_005",
        "topic": "Ford F-150 — Full-Size Pickup Truck",
        "text": """The Ford F-150 is America's best-selling vehicle for 46+ consecutive years.
Starting price: approximately $36,080 USD (Regular Cab XL). Available in Regular, SuperCab, and SuperCrew cab styles.
Engine choices: 2.7L EcoBoost V6 (325 hp), 3.5L EcoBoost V6 (400 hp), 5.0L V8 (400 hp),
3.5L PowerBoost Hybrid (430 hp, 12,700 lb towing), and 1.5L EcoBoost I3.
Towing capacity: up to 14,000 lbs (3.5L EcoBoost with Max Trailer Tow Package).
Payload: up to 2,455 lbs. Bed lengths: 5.5, 6.5, or 8 feet.
Fuel economy (PowerBoost Hybrid): 25 MPG city / 26 MPG highway.
The F-150 Lightning is the electric variant — up to 320 miles range, 775 lb-ft torque, 10,000 lb towing (Max Tow).
Pro Power Onboard available (up to 9.6 kW of exportable power for jobsite tools).
NHTSA: 5-star (SuperCrew 4WD). Best for buyers needing serious work, towing, and hauling capability."""
    },
    {
        "id": "doc_006",
        "topic": "Hyundai Ioniq 6 — Long-Range Electric Sedan",
        "text": """The Hyundai Ioniq 6 is a sleek electric sedan built on the E-GMP platform.
Starting price: ~$38,615 USD (SE Standard Range RWD). Long Range RWD tops out at ~$45,900 USD.
Battery options: 53 kWh (Standard Range, 151 miles) and 77.4 kWh (Long Range, up to 361 miles RWD — best-in-class range).
Charging: 800V architecture supports 350 kW DC fast charging — 10% to 80% in about 18 minutes.
Performance: Long Range AWD — 320 hp, 0-60 in 5.1 sec. SE RWD — 149 hp.
Aerodynamics: drag coefficient of 0.21 Cd — one of the lowest in production.
Interior: 12-inch touchscreen + 12-inch digital cluster, flat floor (great for rear passengers), vehicle-to-load (V2L) feature.
NHTSA: 5-star overall. IIHS Top Safety Pick+.
Eligible for $7,500 federal EV tax credit (check eligibility). Warranty: 5 years/60,000 basic; 10 years/100,000 battery.
Best for EV buyers who prioritize maximum range, fast charging, and value pricing."""
    },
    {
        "id": "doc_007",
        "topic": "Toyota RAV4 — Best-Selling SUV Specs",
        "text": """The Toyota RAV4 is the world's best-selling SUV. Starting at ~$28,975 USD (LE FWD).
Trims: LE, XLE, XLE Premium, TRD Off-Road, Adventure, Limited. AWD available on all but base LE.
Engine: 2.5L 4-cylinder (203 hp). Fuel economy: 27 MPG city / 35 MPG highway (AWD).
RAV4 Hybrid (starts ~$32,150): 219 hp combined, 38 MPG city / 38 MPG highway — exceptional efficiency.
RAV4 Prime PHEV: 302 hp, 42 miles electric-only range, then operates as a full hybrid (38 MPGe combined).
Cargo space: 37.6 cu ft behind rear seats; 69.8 cu ft max. Towing: up to 3,500 lbs (non-hybrid with tow package).
Toyota Safety Sense 2.0 standard on all trims. NHTSA: 5-star. IIHS Top Safety Pick+.
Ground clearance: 8.6 inches (Adventure/TRD) for light off-roading.
Excellent resale value and reliability history make the RAV4 one of the safest financial choices in the SUV segment."""
    },
    {
        "id": "doc_008",
        "topic": "Porsche 911 — Sports Car Performance",
        "text": """The Porsche 911 is the definitive sports car, in production since 1963. Starting at ~$115,000 USD (Carrera).
Variants: Carrera, Carrera S, Carrera 4, Carrera 4S, Targa 4, GTS, GT3, GT3 RS, Turbo, Turbo S.
Engine: Rear-mounted 3.0L twin-turbo flat-6 (379 hp in Carrera; 473 hp in Carrera S). GT3 uses naturally aspirated 4.0L (502 hp).
Turbo S: 640 hp, 0-60 in 2.6 seconds, top speed 205 mph. GT3 RS: 518 hp, Nürburgring lap: 6 min 49.3 sec.
Transmission: 8-speed PDK dual-clutch (manual available on select models).
Fuel economy (Carrera): 18 MPG city / 25 MPG highway. The 911 is daily-drivable yet track-capable.
Interior: 10.9-inch touchscreen, sport seats, leather-wrapped everything. Seats 4 (2+2 layout).
Porsche Active Suspension Management (PASM) standard. Porsche Stability Management standard.
Resale value is exceptional — 911s depreciate very slowly compared to competitors.
Best for driving purists who want a car that can be commuted in on Monday and track-driven on Saturday."""
    },
    {
        "id": "doc_009",
        "topic": "Car Safety Ratings — How to Read Them",
        "text": """Two major organizations rate car safety in the US: NHTSA and IIHS.
NHTSA (National Highway Traffic Safety Administration) uses a 5-star system across:
frontal crash, side crash, rollover. 5 stars = best. Look for 5-star overall for top safety.
IIHS (Insurance Institute for Highway Safety) uses Good / Acceptable / Marginal / Poor ratings in:
small overlap front, moderate overlap front, side (updated), roof strength, head restraints & seats.
IIHS Top Safety Pick (TSP) = passes all crashworthiness tests + acceptable/good on at least one headlight.
IIHS Top Safety Pick+ (TSP+) = TSP + good headlights on all trims. TSP+ is the gold standard.
Euro NCAP operates similarly in Europe using a 0-5 star rating across adult, child, pedestrian, and safety assist categories.
Active safety systems to look for: Automatic Emergency Braking (AEB), Blind Spot Monitoring (BSM),
Lane Keeping Assist (LKA), Rear Cross Traffic Alert (RCTA), and adaptive cruise control.
Always cross-reference both NHTSA and IIHS ratings for the specific model year you are buying."""
    },
    {
        "id": "doc_010",
        "topic": "EV Buying Guide — Key Factors",
        "text": """When buying an electric vehicle (EV), consider these key factors:
1. Range: EPA-rated range varies by trim and conditions. Cold weather reduces range by 20-40%.
   Aim for 50-100 miles more than your typical daily need to handle weather and charging gaps.
2. Charging infrastructure: Check DCFC (DC Fast Charging) availability on your routes.
   Tesla Supercharger network is largest in the US. Other EVs use CCS or CHAdeMO connectors.
   As of 2025, most major automakers (Ford, GM, Honda, Toyota) have adopted NACS (Tesla) connector.
3. Home charging: Level 1 (120V outlet) adds ~4 miles/hour. Level 2 (240V) adds 15-37 miles/hour.
   Estimate $800-$2,500 for Level 2 home charger installation.
4. Federal tax credit: Up to $7,500 for new EVs under Clean Vehicle Credit (IRA 2022).
   Income limits apply ($150,000 single / $300,000 joint). MSRP caps: $55,000 sedan / $80,000 SUV/truck.
5. Battery warranty: Look for at least 8 years / 100,000 miles battery warranty.
6. Real-world efficiency: Check kWh per 100 miles. Lower = more efficient.
   Ioniq 6 LR: 25 kWh/100 mi. Model 3 LR: 25 kWh/100 mi. Rivian R1T: 35 kWh/100 mi."""
    },
    {
        "id": "doc_011",
        "topic": "Car Maintenance Costs — Comparison by Type",
        "text": """Annual maintenance costs vary significantly by car type and brand:
EVs: Average $550-$900/year. No oil changes, no spark plugs, fewer brake replacements (regenerative braking).
Main costs: tire rotation, cabin air filter, wiper blades, brake fluid, eventual 12V battery replacement.
Japanese brands (Toyota, Honda): Average $400-$700/year. Renowned for reliability and affordable parts.
German luxury brands (BMW, Mercedes, Audi): Average $1,000-$1,800/year. Complex engineering, expensive parts,
specialized service requirements. Often require premium synthetic oil ($150-$250/change).
American trucks (Ford F-150, Chevy Silverado): Average $700-$950/year.
Full brake service (pads + rotors, all 4 wheels): $400-$900 typical.
Transmission service: $150-$300 for fluid change; $1,500-$4,000 for rebuild.
Tire replacement (set of 4): $400-$1,200 depending on size and brand.
Use RepairPal and Consumer Reports for model-specific reliability scores before purchasing.
A Toyota RAV4 costs significantly less to maintain over 10 years than a BMW X3 of similar class."""
    },
    {
        "id": "doc_012",
        "topic": "Car Buying Tips — New vs Used, Negotiation",
        "text": """Key tips for buying a car wisely:
New vs Used: New cars offer warranties, latest tech, and incentives but depreciate 15-25% in year one.
Certified Pre-Owned (CPO) vehicles offer manufacturer warranties and are inspected — best of both worlds.
Best time to buy: End of month, end of quarter (March, June, September, December), or holiday weekends
when dealers push to hit sales targets.
Know the invoice price: Use TrueCar, Edmunds, or CarGurus to find dealer invoice price before negotiating.
Target 2-4% above invoice for popular models; at or below invoice during slow sales periods.
Finance wisely: Get pre-approved by your bank or credit union before visiting a dealer.
Interest rates on auto loans vary: credit scores 750+ typically get under 5% APR (2024 averages).
Avoid extending loan term beyond 60 months to prevent being 'underwater' (owing more than car is worth).
Test drive checklist: Check all tech features, seat comfort, visibility, brake feel, acceleration.
Ask for CarFax or AutoCheck report on used cars. Inspect for rust, uneven panel gaps (accident signs).
Total cost of ownership matters more than sticker price: include insurance, fuel, maintenance, and depreciation."""
    }
]

# ── Build ChromaDB ─────────────────────────────────────────
print("Loading embedding model...")
embedder = SentenceTransformer("all-MiniLM-L6-v2")

client = chromadb.Client()
try:
    client.delete_collection("cars_kb")
except:
    pass
collection = client.create_collection("cars_kb")

texts      = [d["text"]  for d in DOCUMENTS]
ids        = [d["id"]    for d in DOCUMENTS]
embeddings = embedder.encode(texts).tolist()

collection.add(
    documents=texts,
    embeddings=embeddings,
    ids=ids,
    metadatas=[{"topic": d["topic"]} for d in DOCUMENTS]
)

print(f"✅ Cars knowledge base ready: {collection.count()} documents")
for d in DOCUMENTS:
    print(f"   • {d['topic']}")

In [ ]:
# ── Test retrieval before building the graph ──────────────
test_query = "What is the fuel economy of the Toyota Camry Hybrid?"

q_emb   = embedder.encode([test_query]).tolist()
results = collection.query(query_embeddings=q_emb, n_results=3)

print(f"Query: {test_query}")
print(f"\nTop 3 retrieved chunks:")
for i, (doc, meta) in enumerate(zip(results["documents"][0], results["metadatas"][0])):
    print(f"\n[{i+1}] Topic: {meta['topic']}")
    print(f"    Text: {doc[:200]}...")

print("\n✅ If the retrieved chunks are relevant — retrieval is working correctly.")

---
## Part 2 — State Design

In [ ]:
class CarsAgentState(TypedDict):
    # ── Input ──────────────────────────────────────────────
    question:      str          # user's current question

    # ── Memory ─────────────────────────────────────────────
    messages:      List[dict]   # conversation history

    # ── Routing ────────────────────────────────────────────
    route:         str          # "retrieve", "memory_only", "tool"

    # ── RAG ────────────────────────────────────────────────
    retrieved:     str          # ChromaDB context chunks
    sources:       List[str]    # source topic names

    # ── Tool ───────────────────────────────────────────────
    tool_result:   str          # output from web search

    # ── Answer ─────────────────────────────────────────────
    answer:        str          # final LLM response

    # ── Quality control ────────────────────────────────────
    faithfulness:  float        # eval score 0.0-1.0
    eval_retries:  int          # safety valve counter

    # ── Domain-specific ────────────────────────────────────
    car_brand:     str          # extracted brand if detected (e.g. "Toyota", "Tesla")

print("CarsAgentState defined with fields:", list(CarsAgentState.__annotations__.keys()))

---
## Part 3 — Node Functions

In [ ]:
# ── Node 1: Memory ─────────────────────────────────────────
def memory_node(state: CarsAgentState) -> dict:
    msgs = state.get("messages", [])
    msgs = msgs + [{"role": "user", "content": state["question"]}]
    if len(msgs) > 6:  # sliding window: keep last 3 turns
        msgs = msgs[-6:]
    return {"messages": msgs}

# Quick test
test_state = {"question": "What is the range of Tesla Model 3?", "messages": []}
result = memory_node(test_state)
print(f"memory_node test: messages={result['messages']}")
print("✅ memory_node works")

In [ ]:
# ── Node 2: Router ─────────────────────────────────────────
def router_node(state: CarsAgentState) -> dict:
    question = state["question"]
    messages = state.get("messages", [])
    recent   = "; ".join(f"{m['role']}: {m['content'][:60]}" for m in messages[-3:-1]) or "none"

    prompt = f"""
You are a STRICT router for a car assistant system.

You MUST choose exactly ONE route from the following:

1. retrieve → use this for:
   - car specs (MPG, horsepower, range)
   - safety ratings
   - maintenance costs
   - general car info from knowledge base

2. memory_only → use this for:
   - questions about previous conversation
   - e.g. "what did you just say?", "tell me more"

3. tool → use this ONLY for:
   - latest / current / real-time information
   - 2024 or 2025 data
   - prices that may change
   - news or recent updates

Examples:
- "What is Camry MPG?" → retrieve
- "What did you just say?" → memory_only
- "Latest Tesla Model 3 price 2025" → tool

Recent conversation:
{recent}

Current question:
{question}

Respond with ONLY ONE WORD:
retrieve OR memory_only OR tool
"""

    response = llm.invoke(prompt)
    decision = response.content.strip().lower()

    if "memory" in decision:
        decision = "memory_only"
    elif "tool" in decision:
        decision = "tool"
    else:
        decision = "retrieve"

    return {"route": decision}

# Quick test
test_state2 = {"question": "What did you just tell me?", "messages": [{"role":"user","content":"hi"}]}
result2 = router_node(test_state2)
print(f"router_node test: route='{result2['route']}' (expected: memory_only)")

In [ ]:
# ── Node 3: Retrieval ──────────────────────────────────────
def retrieval_node(state: CarsAgentState) -> dict:
    q_emb   = embedder.encode([state["question"]]).tolist()
    results = collection.query(query_embeddings=q_emb, n_results=3)
    chunks  = results["documents"][0]
    topics  = [m["topic"] for m in results["metadatas"][0]]
    context = "\n\n---\n\n".join(f"[{topics[i]}]\n{chunks[i]}" for i in range(len(chunks)))
    return {"retrieved": context, "sources": topics}


def skip_retrieval_node(state: CarsAgentState) -> dict:
    return {"retrieved": "", "sources": []}


# Quick test
test_state3 = {"question": "What is the towing capacity of the Ford F-150?"}
result3 = retrieval_node(test_state3)
print(f"retrieval_node test: sources={result3['sources']}")
print(f"  Context preview: {result3['retrieved'][:200]}...")
print("✅ retrieval_node works")

In [ ]:
# ── Node 4: Tool — DuckDuckGo Web Search ───────────────────
# Used for live pricing, breaking news, latest car releases

# ── Node 4: Tool — DuckDuckGo Web Search ───────────────────
def tool_node(state: CarsAgentState) -> dict:
    """Web search for live car prices, news, and latest releases."""
    question = state["question"]

    try:
        from duckduckgo_search import DDGS

        # 🔥 Improved query
        query = f"{question} latest car price 2025 news"

        with DDGS() as ddgs:
            results = list(ddgs.text(query, max_results=5))

        if not results:
            return {
                "tool_result": "No live results found. Please check official manufacturer websites or Edmunds."
            }

        # 🔥 Better formatting for LLM
        formatted = "\n\n".join(
            f"Title: {r['title']}\nSummary: {r['body'][:300]}"
            for r in results
        )

        return {"tool_result": formatted}

    except ImportError:
        return {
            "tool_result": "Web search not available. Install with: pip install duckduckgo-search"
        }

    except Exception as e:
        return {
            "tool_result": f"Web search error: {e}. Try checking official sites like Tesla or Edmunds."
        }

# Quick test
test_tool_state = {"question": "What is the current price of Toyota RAV4 2025?"}
tool_out = tool_node(test_tool_state)
print(f"tool_node test output preview: {tool_out['tool_result'][:300]}")
print("✅ tool_node works")

In [ ]:
# ── Node 5: Answer ─────────────────────────────────────────
def answer_node(state: CarsAgentState) -> dict:
    retrieved = state.get("retrieved", "")
    tool_result = state.get("tool_result", "")
    eval_retries = state.get("eval_retries", 0)

    # 🔗 Combine context
    context_parts = []

    if retrieved:
        context_parts.append(f"KNOWLEDGE BASE:\n{retrieved}")

    if tool_result:
        context_parts.append(f"WEB SEARCH RESULTS:\n{tool_result}")

    context = "\n\n".join(context_parts)

    # ✅ Clean and correct prompt
    if context:
        system_prompt = f"""
You are an expert automotive assistant.

Use the provided context (knowledge base or web results) to answer the question.

If context exists:
- Extract useful details
- Summarize clearly
- Do NOT refuse unnecessarily

If NO relevant information exists in the context:
- Then say: "I don't have that specific information — please check official sources."

CONTEXT:
{context}
"""
    else:
        # memory-only case
        system_prompt = "You are a helpful automotive assistant. Answer based on the conversation history."

    # 🔁 Retry logic (keep strict)
    if eval_retries > 0:
        system_prompt += "\n\nIMPORTANT: Do NOT add information not present in the context."

    # 📩 Build messages
    msgs = [{"role": "system", "content": system_prompt}]
    msgs += state.get("messages", [])

    response = llm.invoke(msgs)

    return {"answer": response.content}

In [ ]:
# ── Node 6: Eval — faithfulness scoring ────────────────────
FAITHFULNESS_THRESHOLD = 0.7
MAX_EVAL_RETRIES       = 2

def eval_node(state: CarsAgentState) -> dict:
    answer  = state.get("answer", "")
    context = state.get("retrieved", "")[:500]
    retries = state.get("eval_retries", 0)

    if not context:
        return {"faithfulness": 1.0, "eval_retries": retries + 1}

    prompt = f"""Rate faithfulness: does this car-related answer use ONLY information from the context?
Reply with ONLY a number between 0.0 and 1.0.
1.0 = fully faithful to context. 0.5 = some hallucination. 0.0 = mostly hallucinated.

Context: {context}
Answer: {answer[:300]}"""

    result = llm.invoke(prompt).content.strip()
    try:
        score = float(result.split()[0].replace(",", "."))
        score = max(0.0, min(1.0, score))
    except:
        score = 0.5

    gate = "✅" if score >= FAITHFULNESS_THRESHOLD else "⚠️"
    print(f"  [eval] Faithfulness: {score:.2f} {gate}")
    return {"faithfulness": score, "eval_retries": retries + 1}

# ── Node 7: Save ───────────────────────────────────────────
def save_node(state: CarsAgentState) -> dict:
    messages = state.get("messages", []).copy()
    answer = state.get("answer", "").strip()

    # ❗ Skip empty or weak fallback responses
    if answer and not any(
        phrase in answer.lower()
        for phrase in [
            "i don't have that specific information",
            "i don't have enough information",
            "no relevant information found"
        ]
    ):
        messages.append({
            "role": "assistant",
            "content": answer
        })

    return {"messages": messages}

---
## Part 4 — Graph Assembly

In [ ]:
# ── Routing functions ──────────────────────────────────────

def route_decision(state: CarsAgentState) -> str:
    """After router_node: decide which path."""
    route = state.get("route", "retrieve")
    if route == "tool":        return "tool"
    if route == "memory_only": return "skip"
    return "retrieve"


def eval_decision(state: CarsAgentState) -> str:
    """After eval_node: retry or save."""
    score   = state.get("faithfulness", 1.0)
    retries = state.get("eval_retries", 0)
    if score >= FAITHFULNESS_THRESHOLD or retries >= MAX_EVAL_RETRIES:
        return "save"
    return "answer"  # retry


# ── Build the graph ────────────────────────────────────────
graph = StateGraph(CarsAgentState)

graph.add_node("memory",   memory_node)
graph.add_node("router",   router_node)
graph.add_node("retrieve", retrieval_node)
graph.add_node("skip",     skip_retrieval_node)
graph.add_node("tool",     tool_node)
graph.add_node("answer",   answer_node)
graph.add_node("eval",     eval_node)
graph.add_node("save",     save_node)

graph.set_entry_point("memory")
graph.add_edge("memory", "router")

graph.add_conditional_edges(
    "router", route_decision,
    {"retrieve": "retrieve", "skip": "skip", "tool": "tool"}
)

graph.add_edge("retrieve", "answer")
graph.add_edge("skip",     "answer")
graph.add_edge("tool",     "answer")

graph.add_edge("answer", "eval")
graph.add_conditional_edges(
    "eval", eval_decision,
    {"answer": "answer", "save": "save"}
)
graph.add_edge("save", END)

checkpointer = MemorySaver()
app = graph.compile(checkpointer=checkpointer)

print("✅ Cars Agent Graph compiled!")
print("Flow: memory → router → [retrieve | skip | tool] → answer → eval → save → END")

---
## Part 5 — Testing (10 Questions + 2 Red-Team)

In [ ]:
def ask(question: str, thread_id: str = "test") -> dict:
    """Helper to run the cars agent and return result."""
    config = {"configurable": {"thread_id": thread_id}}
    result = app.invoke({"question": question}, config=config)
    return result


TEST_QUESTIONS = [
    # Domain questions
    {"q": "What is the fuel economy of the Toyota Camry Hybrid?",
     "expect": "Should mention 51 MPG city / 53 MPG highway", "red_team": False},

    {"q": "How many miles of range does the Tesla Model 3 Long Range get?",
     "expect": "Should say 358 miles", "red_team": False},

    {"q": "What is the towing capacity of the Ford F-150 with the Max Trailer Tow Package?",
     "expect": "Should say 14,000 lbs", "red_team": False},

    {"q": "What does IIHS Top Safety Pick+ mean?",
     "expect": "Should explain TSP+ = passes crashworthiness + good headlights", "red_team": False},

    {"q": "Compare the Honda CR-V Hybrid and the Toyota RAV4 Hybrid fuel economy.",
     "expect": "Should compare CR-V Hybrid (40/34) vs RAV4 Hybrid (38/38)", "red_team": False},

    {"q": "What are the annual maintenance costs for a BMW versus a Toyota?",
     "expect": "Should mention BMW $1,000-$1,800 vs Toyota $400-$700", "red_team": False},

    {"q": "What is the 0-60 time of the Porsche 911 Turbo S?",
     "expect": "Should say 2.6 seconds", "red_team": False},

    {"q": "What EV charging level should I install at home?",
     "expect": "Should recommend Level 2 (240V), mention 15-37 miles/hour", "red_team": False},

    # Red-team tests
    {"q": "What is the fuel economy of a Lamborghini Urus?",
     "expect": "Should admit this info is not in the knowledge base", "red_team": True},

    {"q": "Isn't the Tesla Model 3 the fastest production car in the world at 0-60 in 1.5 seconds?",
     "expect": "Should correct the false premise — Model 3 Performance is 3.1 sec, not 1.5 sec", "red_team": True},
]

print(f"Prepared {len(TEST_QUESTIONS)} test questions ({sum(1 for t in TEST_QUESTIONS if t['red_team'])} red-team)")

In [ ]:
test_results = []

print("=" * 65)
print("RUNNING CARS AGENT TEST SUITE")
print("=" * 65)

for i, test in enumerate(TEST_QUESTIONS):
    print(f"\n--- Test {i+1} {'[RED TEAM 🔴]' if test['red_team'] else ''} ---")
    print(f"Q: {test['q']}")

    result = ask(test["q"], thread_id=f"test-{i}")
    answer = result.get("answer", "")
    faith  = result.get("faithfulness", 0.0)
    route  = result.get("route", "?")

    print(f"A: {answer[:300]}")
    print(f"Route: {route} | Faithfulness: {faith:.2f}")
    print(f"Expected: {test['expect']}")

    # Pass if answer is substantive and not empty
    passed = len(answer) > 30 and "error" not in answer.lower()

    print(f"Result: {'✅ PASS' if passed else '❌ FAIL'}")
    test_results.append({
        "q": test["q"][:55], "passed": passed,
        "faith": faith, "route": route, "red_team": test["red_team"]
    })

# Summary
total  = len(test_results)
passed = sum(1 for r in test_results if r["passed"])
print(f"\n{'='*65}")
print(f"RESULTS: {passed}/{total} passed")
print(f"Average faithfulness: {sum(r['faith'] for r in test_results)/total:.2f}")

---
## Part 6 — RAGAS Baseline Evaluation

In [ ]:
RAGAS_QUESTIONS = [
    {
        "question": "What is the fuel economy of the Toyota Camry Hybrid?",
        "ground_truth": "The Toyota Camry Hybrid achieves up to 51 MPG city and 53 MPG highway."
    },
    {
        "question": "How much does the Tesla Model 3 Standard Range cost?",
        "ground_truth": "The Tesla Model 3 Standard Range RWD starts at approximately $40,240 USD before incentives."
    },
    {
        "question": "What is the cargo space of the Honda CR-V?",
        "ground_truth": "The Honda CR-V has 39.3 cubic feet of cargo space behind the rear seats and 76.5 cubic feet with the rear seats folded."
    },
    {
        "question": "What does IIHS Top Safety Pick+ mean?",
        "ground_truth": "IIHS Top Safety Pick+ means the vehicle passes all crashworthiness tests and has good-rated headlights on all trims."
    },
    {
        "question": "What is the 0-60 time of the Porsche 911 Turbo S?",
        "ground_truth": "The Porsche 911 Turbo S achieves 0-60 mph in 2.6 seconds and has a top speed of 205 mph."
    },
]

eval_dataset = []
print("Running cars agent for RAGAS evaluation...")
for rq in RAGAS_QUESTIONS:
    q_emb   = embedder.encode([rq["question"]]).tolist()
    results = collection.query(query_embeddings=q_emb, n_results=3)
    chunks  = results["documents"][0]
    result  = ask(rq["question"], thread_id=f"ragas-{rq['question'][:10]}")
    eval_dataset.append({
        "question":     rq["question"],
        "answer":       result.get("answer", ""),
        "contexts":     chunks,
        "ground_truth": rq["ground_truth"]
    })
    print(f"  ✓ {rq['question'][:60]}")

print(f"\n✅ Eval dataset built: {len(eval_dataset)} rows")

In [ ]:
try:
    from ragas import evaluate
    from ragas.metrics import faithfulness, answer_relevancy, context_precision
    from datasets import Dataset

    ragas_data   = Dataset.from_list(eval_dataset)
    print("Running RAGAS evaluation (1-2 minutes)...")

    ragas_result = evaluate(
        dataset=ragas_data,
        metrics=[faithfulness, answer_relevancy, context_precision],
    )

    df = ragas_result.to_pandas()
    print("\n" + "=" * 45)
    print("BASELINE RAGAS SCORES — CARS AGENT")
    print("=" * 45)
    print(f"Faithfulness:      {df['faithfulness'].mean():.3f}")
    print(f"Answer Relevance:  {df['answer_relevancy'].mean():.3f}")
    print(f"Context Precision: {df['context_precision'].mean():.3f}")
    print("\n⚠️  Record these baseline scores. Re-run after improvements.")

except ImportError:
    print("RAGAS not installed — running manual faithfulness scoring")
    faith_scores = []
    for row in eval_dataset:
        prompt = f"""Rate faithfulness 0.0-1.0. Reply with only a number.
Context: {row['contexts'][0][:300]}
Answer: {row['answer'][:200]}"""
        try:
            score = float(llm.invoke(prompt).content.strip().split()[0])
            score = max(0.0, min(1.0, score))
        except:
            score = 0.5
        faith_scores.append(score)
        print(f"  Q: {row['question'][:50]:50s} → {score:.2f}")

    avg = sum(faith_scores) / len(faith_scores)
    print(f"\nBaseline faithfulness: {avg:.3f}")
    print("Install RAGAS for full evaluation: pip install ragas datasets")

---
## Part 7 — Deployment: Streamlit App

In [ ]:
streamlit_code = '''
"""
cars_agent_streamlit.py — Cars Production Agent UI
Run: streamlit run cars_agent_streamlit.py
"""
import streamlit as st
import uuid
import os
import chromadb
from dotenv import load_dotenv
from typing import TypedDict, List
from sentence_transformers import SentenceTransformer
from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver

load_dotenv()

st.set_page_config(page_title="🚗 Cars Expert Agent", page_icon="🚗", layout="centered")
st.title("🚗 Cars Expert Agent")
st.caption("Ask me anything about car specs, pricing, safety ratings, EVs, and buying tips!")

DOCUMENTS = [
    {"id":"doc_001","topic":"Toyota Camry","text":"""The Toyota Camry is a mid-size sedan starting at ~$27,000. Hybrid gets 51/53 MPG. 5-star NHTSA. IIHS TSP+."""},
    {"id":"doc_002","topic":"Tesla Model 3","text":"""Tesla Model 3 starts at ~$40,240. Long Range = 358 miles. Performance does 0-60 in 3.1 sec. NHTSA 5-star."""},
    {"id":"doc_003","topic":"Honda CR-V","text":"""Honda CR-V starts at ~$31,895. Hybrid gets 40/34 MPG. 39.3 cu ft cargo. Tows 1,500 lbs. IIHS TSP+."""},
    {"id":"doc_004","topic":"BMW 3 Series","text":"""BMW 3 Series starts at ~$45,000. M340i: 382 hp, 0-60 in 4.2 sec. Annual maintenance ~$1,000-$1,800."""},
    {"id":"doc_005","topic":"Ford F-150","text":"""Ford F-150 starts at ~$36,080. Tows up to 14,000 lbs. PowerBoost Hybrid: 430 hp, 25/26 MPG. NHTSA 5-star."""},
    {"id":"doc_006","topic":"Hyundai Ioniq 6","text":"""Ioniq 6 starts at ~$38,615. Long Range RWD: 361 miles. 800V charging: 10-80% in 18 min. IIHS TSP+."""},
    {"id":"doc_007","topic":"Toyota RAV4","text":"""RAV4 starts at ~$28,975. Hybrid: 38/38 MPG. Prime PHEV: 42 miles EV range. Tows 3,500 lbs."""},
    {"id":"doc_008","topic":"Porsche 911","text":"""Porsche 911 starts at ~$115,000. Turbo S: 640 hp, 0-60 in 2.6 sec, top speed 205 mph. Exceptional resale value."""},
    {"id":"doc_009","topic":"Car Safety Ratings","text":"""NHTSA uses 5-star system. IIHS TSP = passes crashworthiness. TSP+ = TSP plus good headlights. Best is NHTSA 5-star + IIHS TSP+."""},
    {"id":"doc_010","topic":"EV Buying Guide","text":"""For EVs: check range, charging network, home charging (Level 2 = 15-37 mi/hr), federal tax credit up to $7,500, battery warranty 8yr/100k."""},
    {"id":"doc_011","topic":"Maintenance Costs","text":"""EVs: $550-$900/yr. Toyota/Honda: $400-$700/yr. BMW/Mercedes: $1,000-$1,800/yr. Ford trucks: $700-$950/yr."""},
    {"id":"doc_012","topic":"Car Buying Tips","text":"""Best time to buy: end of month/quarter. Get bank pre-approval. Target 2-4% above invoice. Avoid loans over 60 months. Consider CPO for used."""},
]

FAITHFULNESS_THRESHOLD = 0.7
MAX_EVAL_RETRIES = 2

@st.cache_resource
def load_agent():
    llm      = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)
    embedder = SentenceTransformer("all-MiniLM-L6-v2")

    client = chromadb.Client()
    try: client.delete_collection("cars_kb")
    except: pass
    collection = client.create_collection("cars_kb")
    texts = [d["text"] for d in DOCUMENTS]
    collection.add(
        documents=texts,
        embeddings=embedder.encode(texts).tolist(),
        ids=[d["id"] for d in DOCUMENTS],
        metadatas=[{"topic": d["topic"]} for d in DOCUMENTS]
    )

    class CarsAgentState(TypedDict):
        question: str; messages: List[dict]; route: str
        retrieved: str; sources: List[str]; tool_result: str
        answer: str; faithfulness: float; eval_retries: int; car_brand: str

    def memory_node(state):
        msgs = state.get("messages", []) + [{"role":"user","content":state["question"]}]
        return {"messages": msgs[-6:]}

    def router_node(state):
        prompt = f"""Router for a car info chatbot. Options: retrieve (specs/safety/pricing in KB), memory_only (follow-up), tool (live prices/news).\nQuestion: {state[\"question\"]}\nReply ONE word: retrieve/memory_only/tool"""
        d = llm.invoke(prompt).content.strip().lower()
        if "memory" in d: d = "memory_only"
        elif "tool" in d: d = "tool"
        else: d = "retrieve"
        return {"route": d}

    def retrieval_node(state):
        q_emb = embedder.encode([state["question"]]).tolist()
        res = collection.query(query_embeddings=q_emb, n_results=3)
        topics = [m["topic"] for m in res["metadatas"][0]]
        ctx = "\n\n---\n\n".join(f"[{topics[i]}]\n{res['documents'][0][i]}" for i in range(len(topics)))
        return {"retrieved": ctx, "sources": topics}

    def skip_retrieval_node(state): return {"retrieved": "", "sources": []}

    def tool_node(state):
        try:
            from duckduckgo_search import DDGS
            with DDGS() as ddgs:
                results = list(ddgs.text(state["question"] + " car 2025", max_results=3))
            return {"tool_result": "\n".join(f"{r['title']}: {r['body'][:200]}" for r in results)}
        except Exception as e:
            return {"tool_result": f"Web search unavailable: {e}. Check Edmunds or CarGurus for live data."}

    def answer_node(state):
        retrieved = state.get("retrieved","")
        tool_result = state.get("tool_result","")
        ctx_parts = []
        if retrieved: ctx_parts.append(f"KNOWLEDGE BASE:\n{retrieved}")
        if tool_result: ctx_parts.append(f"WEB SEARCH:\n{tool_result}")
        ctx = "\n\n".join(ctx_parts)
        sys = f"You are an expert car advisor. Use ONLY the provided context. If not found, say so.\n\n{ctx}" if ctx else "You are a car advisor. Answer from conversation history."
        msgs = [SystemMessage(content=sys)]
        for m in state.get("messages",[]):
            msgs.append(HumanMessage(content=m["content"]) if m["role"]=="user" else AIMessage(content=m["content"]))
        return {"answer": llm.invoke(msgs).content}

    def eval_node(state):
        ctx = state.get("retrieved","")[:400]
        retries = state.get("eval_retries", 0)
        if not ctx: return {"faithfulness": 1.0, "eval_retries": retries+1}
        prompt = f"Rate faithfulness 0.0-1.0. Only number.\nContext: {ctx}\nAnswer: {state.get('answer','')[:200]}"
        try: score = max(0.0, min(1.0, float(llm.invoke(prompt).content.strip().split()[0])))
        except: score = 0.5
        return {"faithfulness": score, "eval_retries": retries+1}

    def save_node(state):
        msgs = state.get("messages",[]) + [{"role":"assistant","content":state["answer"]}]
        return {"messages": msgs}

    def route_dec(state):
        r = state.get("route","retrieve")
        if r=="tool": return "tool"
        if r=="memory_only": return "skip"
        return "retrieve"

    def eval_dec(state):
        if state.get("faithfulness",1.0) >= FAITHFULNESS_THRESHOLD or state.get("eval_retries",0) >= MAX_EVAL_RETRIES:
            return "save"
        return "answer"

    g = StateGraph(CarsAgentState)
    for name, fn in [("memory",memory_node),("router",router_node),("retrieve",retrieval_node),
                     ("skip",skip_retrieval_node),("tool",tool_node),("answer",answer_node),("eval",eval_node),("save",save_node)]:
        g.add_node(name, fn)
    g.set_entry_point("memory")
    g.add_edge("memory","router")
    g.add_conditional_edges("router", route_dec, {"retrieve":"retrieve","skip":"skip","tool":"tool"})
    for n in ["retrieve","skip","tool"]: g.add_edge(n,"answer")
    g.add_edge("answer","eval")
    g.add_conditional_edges("eval", eval_dec, {"answer":"answer","save":"save"})
    g.add_edge("save", END)
    agent_app = g.compile(checkpointer=MemorySaver())
    return agent_app, embedder, collection

try:
    agent_app, embedder, collection = load_agent()
    st.success(f"✅ Cars knowledge base loaded — {collection.count()} documents")
except Exception as e:
    st.error(f"Failed to load agent: {e}")
    st.stop()

if "messages" not in st.session_state:
    st.session_state.messages = []
if "thread_id" not in st.session_state:
    st.session_state.thread_id = str(uuid.uuid4())[:8]

with st.sidebar:
    st.header("🚗 About")
    st.write("Expert car advisor covering specs, pricing, safety ratings, EVs, maintenance costs, and buying tips.")
    st.write(f"Session: {st.session_state.thread_id}")
    st.divider()
    st.write("**Topics covered:**")
    topics = [d["topic"] for d in DOCUMENTS]
    for t in topics:
        st.write(f"• {t}")
    if st.button("🗑️ New conversation"):
        st.session_state.messages = []
        st.session_state.thread_id = str(uuid.uuid4())[:8]
        st.rerun()

for msg in st.session_state.messages:
    with st.chat_message(msg["role"]):
        st.write(msg["content"])

if prompt := st.chat_input("Ask about any car — specs, price, safety, EV range, buying tips..."):
    with st.chat_message("user"):
        st.write(prompt)
    st.session_state.messages.append({"role":"user","content":prompt})

    with st.chat_message("assistant"):
        with st.spinner("Researching your question..."):
            config = {"configurable": {"thread_id": st.session_state.thread_id}}
            result = agent_app.invoke({"question": prompt}, config=config)
            answer = result.get("answer", "Sorry, I could not generate an answer.")
        st.write(answer)
        faith = result.get("faithfulness", 0.0)
        sources = result.get("sources", [])
        if faith > 0:
            st.caption(f"Faithfulness: {faith:.2f} | Route: {result.get('route','?')} | Sources: {sources}")

    st.session_state.messages.append({"role":"assistant","content":answer})
'''

with open("cars_agent_streamlit.py", "w") as f:
    f.write(streamlit_code.strip())

print("✅ cars_agent_streamlit.py written successfully!")
print()
print("To run the Streamlit app:")
print("  streamlit run cars_agent_streamlit.py")

---
## Part 8 — Written Summary

## My Capstone Summary

**Domain chosen:** Automotive — Car Specifications, Pricing, Safety & Buying Guide

**What the agent does:** The Cars Production Agent helps car buyers, enthusiasts, and researchers get instant, accurate answers about car specifications (engine, horsepower, range, cargo space), pricing, fuel economy, safety ratings (NHTSA/IIHS), EV buying considerations, maintenance costs, and car buying strategies. It uses a 12-document knowledge base for static information and routes live pricing/news queries to DuckDuckGo web search.

**Knowledge base:** 12 documents covering Toyota Camry, Tesla Model 3, Honda CR-V, BMW 3 Series, Ford F-150, Hyundai Ioniq 6, Toyota RAV4, Porsche 911, Car Safety Ratings, EV Buying Guide, Maintenance Costs, and Car Buying Tips.

**Tool used:** DuckDuckGo web search (`duckduckgo-search`) — essential for live car prices, 2025 model launches, ongoing incentives, and dealer offers that change frequently and cannot be kept current in a static knowledge base.

**RAGAS baseline scores:**
- Faithfulness: (run eval cell to populate)
- Answer Relevance: (run eval cell to populate)
- Context Precision: (run eval cell to populate)

**Test results:** 10/10 tests designed — run test suite cell to see live pass/fail results.

**One thing I would improve with more time:** I would load real spec sheets and press releases as PDFs (e.g., from manufacturer media sites) using a PDF ingestion pipeline, giving the agent access to all trims and option packages for every model year rather than hand-written summaries.

**Most surprising thing I learned building this:** The router is deceptively important — a weak router sends factual spec questions to web search (which returns unreliable results) and live-pricing questions to the knowledge base (which gives stale data). Prompt engineering the router carefully is as important as building the RAG pipeline itself.

---
## ✅ Submission Checklist

- [x] All TODO sections filled in
- [x] Knowledge base has 12 documents (minimum 10)
- [x] All cells run top-to-bottom without errors
- [x] Test suite covers 10 questions (8 domain + 2 red-team)
- [x] RAGAS evaluation cell ready (populate scores after running)
- [x] `cars_agent_streamlit.py` generated and ready to run
- [x] Conversation memory implemented (MemorySaver + thread_id)
- [x] Tool use: DuckDuckGo web search for live pricing/news
- [x] Self-reflection: faithfulness eval node with retry loop
- [x] Written summary complete

**Deliverables:**
1. `cars_production_agent.ipynb` — this completed notebook
2. `cars_agent_streamlit.py` — generated by Part 7 cell

---
*The Cars Production Agent is ready. Run all cells top-to-bottom, then launch the UI with `streamlit run cars_agent_streamlit.py`.*